# Avaliação dos Modelos de ML/IA — Pipeline Ceará Transparente

Notebook de treinamento e avaliação da Fase 3 (tarefa #22 do cronograma, item 6.1.11 dos entregáveis).

Cobre as 3 peças da seção 4.3 do enunciado:
- **Modelo 1** — Detecção de Anomalias Contratuais (Isolation Forest)
- **Modelo 2** — Previsão de Pagamentos Trimestrais por Órgão (XGBoost, regressão por quantil)
- **Componente de IA Generativa** — Relatório narrativo (LLM via API OpenAI)

> **Nota de proveniência (25/07/2026):** este notebook consolida os dois notebooks anteriores
> (`ml_anomalia_contratos.ipynb` + `ml_previsao_pagamentos.ipynb`) num só, usando o nome do
> notebook original da Fernanda (`eda_e_treinamento_ml.ipynb`) — ela é a dona desta atividade
> (Fase 3/ML, tarefas 20-22). O conteúdo técnico foi atualizado para a versão em produção dos
> módulos (`models/anomaly_detection.py`, `models/payment_forecast.py`,
> `models/narrative_report.py`), incluindo **2 bugs reais corrigidos nesta mesma rodada**,
> encontrados comparando com o script independente que ela enviou:
> 1. `dim_credor.historico_infringement` saía 100% `False` pra todo mundo — o dedup de credor
>    escolhia uma linha por ordem alfabética do nome em vez de agregar infração de TODOS os
>    contratos do credor (`dbt/snapshots/scd_credor.sql`, corrigido com `bool_or`).
> 2. `sk_orgao` muda a cada ano pro MESMO órgão físico (`dim_orgao` é versionada por
>    `(codigo, ano)`) — agrupar a série temporal por `sk_orgao` (como o Modelo 2 fazia)
>    quebrava o histórico plurianual: `lag_4_trimestres` nunca tinha valor real (0 de 14.920
>    linhas em produção) e `valor_contratado_ativo` saía zerado em 89,5% das linhas. Corrigido
>    agrupando por `codigo_orgao` (estável no tempo) — o mesmo problema que a Fernanda achou
>    e corrigiu no script dela (que usa Prophet em vez de XGBoost).

Os runs de treinamento ficam registrados no MLflow — `mlflow ui --backend-store-uri
file://<repo>/models/artifacts/mlruns` pra conferir depois (ver
`documentacao/guia-de-exploracao.md`, seção 4).


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
load_dotenv(dotenv_path=project_root / ".env")

from models import anomaly_detection as ad
from models import payment_forecast as pf
from models import narrative_report as nr

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Modelo 1 — Detecção de Anomalias Contratuais

Treina e avalia o `IsolationForest` de `models/anomaly_detection.py` sobre a Gold real (`iceberg.gold.fato_contrato` + dimensões, via Trino). Não supervisionado: não existe rótulo de "contrato irregular" em nenhuma fonte — o modelo aprende só a partir da distribuição dos próprios dados (ver docstring do módulo, seção "Desbalanceamento", tarefa 28).

Pré-requisito: stack do lakehouse no ar (Hive Metastore + Trino, Gold já construída pelo `dbt build`).

### 1.1 Extração das features (Gold + Silver via Trino)

`fato_contrato` já traz `valor_contrato`/`flag_emergency`; `dim_credor` traz o histórico de infração (SCD2, **corrigido nesta rodada** — antes saía 100% `False`); `dim_modalidade`, a modalidade. `tipo_objeto` e as datas de vigência ainda não estão modeladas na Gold, então a query junta direto com `iceberg.silver.contratos` para pegá-los — ver `anomaly_detection.FEATURE_QUERY`.

In [ ]:
raw = ad.extract_features()
print(f"Contratos com valor_contrato preenchido: {len(raw)}")
raw.head()


In [ ]:
print("Nulos por coluna (%):")
print((raw.isna().mean() * 100).round(1))
print()
print("% flag_emergency=True:", round(100 * raw["flag_emergency"].fillna(False).mean(), 2))
print("% historico_credor_infringement=True:", round(100 * raw["historico_credor_infringement"].fillna(False).mean(), 2))


### 1.2 Engenharia de features

`build_feature_matrix` converte o bruto em matriz numérica: `valor_contrato`, `dias_vigencia` (término − início), one-hot de `modalidade`/`tipo_objeto` (categorias raras agrupadas em `OUTROS`) e as duas flags booleanas.

In [ ]:
X = ad.build_feature_matrix(raw)
print(f"Shape da matriz de features: {X.shape}")
X.describe().T


### 1.3 Treino — Isolation Forest

`contamination="auto"` (padrão do scikit-learn) — não fixamos a priori qual fração dos contratos é anômala (não há rótulo ground-truth pra calibrar isso, dica 7.3 do enunciado). Além do score contínuo, também calculamos `flag_anomalia` (`predict() == -1`, na taxa de `contamination` usada) — classificação binária complementar, incorporada do script da Fernanda.

In [ ]:
model = ad.train_model(X, contamination="auto")
scores = ad.score_anomalia(model, X)
flags = ad.flag_anomalia(model, X)

resultado = raw.copy()
resultado["score_anomalia"] = scores.values
resultado["flag_anomalia"] = flags.values

print(f"Total avaliado: {len(resultado)} | flag_anomalia=True: {int(resultado['flag_anomalia'].sum())} ({resultado['flag_anomalia'].mean():.1%})")
resultado[["id_contrato_origem", "ano", "valor_contrato", "modalidade", "score_anomalia", "flag_anomalia"]].head()


### 1.4 Avaliação

Sem rótulo de verdade, a avaliação é por distribuição do score e checagem de sanidade: os contratos mais anômalos fazem sentido de negócio (valor muito fora da curva, vigência atípica, credor com histórico de infração, contratação emergencial)? Ver também `summarize_score_distribution()` (tarefa 28) para os percentis/thresholds usados na calibração do ponto de corte operacional.

In [ ]:
print("Distribuição do score de anomalia:")
print(resultado["score_anomalia"].describe())
print()
print("Percentis e contagem acima de thresholds (o que fica logado no MLflow a cada run):")
import json as _json
print(_json.dumps(ad.summarize_score_distribution(resultado), indent=2, ensure_ascii=False))


In [ ]:
colunas_contexto = [
    "id_contrato_origem", "ano", "valor_contrato", "modalidade", "tipo_objeto",
    "flag_emergency", "historico_credor_infringement", "score_anomalia", "flag_anomalia",
]
resultado.sort_values("score_anomalia", ascending=False)[colunas_contexto].head(20)


#### Checagem de sanidade — o score reage a sinais de negócio conhecidos?

Contratos emergenciais e credores com histórico de infração deveriam, em média, sair com score mais alto que a base geral — não é garantido (o modelo não sabe o que essas flags significam, só que são incomuns), mas é um sinal de que o modelo captura algo relevante e não só ruído. **Com o bug do `historico_infringement` corrigido**, esta célula agora tem casos reais de credor-com-infração pra comparar (antes, a coluna era 100% `False`).

In [ ]:
comparativo = pd.DataFrame({
    "score_medio_geral": [resultado["score_anomalia"].mean()],
    "score_medio_emergencial": [resultado.loc[resultado["flag_emergency"] == True, "score_anomalia"].mean()],
    "score_medio_credor_com_infracao": [resultado.loc[resultado["historico_credor_infringement"] == True, "score_anomalia"].mean()],
    "n_credores_com_infracao": [int(resultado["historico_credor_infringement"].sum())],
})
comparativo.round(4)


### 1.5 Persistência do modelo

Salva o modelo treinado e a lista de colunas de feature em `models/artifacts/` (git-ignorado) — igual a `python -m models.anomaly_detection`. Além do `.joblib` (usado por `load_model`/`save_model`), a execução via `run()` também loga o modelo no formato nativo do MLflow (`mlflow.sklearn.log_model`, incorporado do script da Fernanda) — habilita Model Registry/serving direto pelo MLflow.

In [ ]:
ad.save_model(model, list(X.columns))
print(f"Modelo salvo em: {ad.ARTIFACT_PATH}")


### Observações para o relatório final — Modelo 1

- [ ] **Total de contratos avaliados / % sinalizado (`flag_anomalia`)** — rode a célula 1.3 acima e preencha com o resultado desta execução.
- [ ] **Padrões nos contratos mais anômalos** (célula 1.4) — checar concentração em `modalidade=DISPENSA`/`flag_emergency=True`, igual observado nas rodadas anteriores contra o DW Postgres legado e a Gold nova.
- [x] **`historico_credor_infringement` — bug corrigido (25/07/2026).** Antes: 100% `False` em toda a base (achado pela Fernanda em dois pipelines independentes — Postgres legado e dbt-trino — o que sugeria característica real da fonte). Investigação mostrou que **não é** característica da fonte: a Silver bruta (`iceberg.silver.contratos.infringement_status`) tem registro real com infração; o bug estava no dedup por credor do `scd_credor.sql`, que escolhia a linha por ordem alfabética do nome em vez de agregar infração de todos os contratos do credor. Corrigido com `bool_or` — ver `dbt/snapshots/scd_credor.sql`.
- [x] **`tipo_objeto`/`dias_vigencia`** (pedidos no enunciado, seção 4.3) — **já estão no modelo**, via `LEFT JOIN` com `iceberg.silver.contratos` (não estão na Gold ainda, mas a Silver tem ambos ~99,9% preenchidos).
- [ ] Levar a lista de contratos mais anômalos para revisão com analistas de controle, se possível (tarefa #35) — pendente.

## 2. Modelo 2 — Previsão de Pagamentos Trimestrais por Órgão

Treina e avalia os regressores por quantil de `models/payment_forecast.py` (XGBoost, `reg:quantileerror`) sobre a Gold real, via Trino.

**Fonte de dado:** `iceberg.gold.fato_ordem_bancaria` — o pagamento efetivo ao credor (3º estágio da despesa: contrato → empenho → ordem bancária). Não é mais proxy (nem `fato_empenho`, nem `fato_contrato.valor_pago` como no script original da Fernanda) — `fato_ordem_bancaria` existe na Gold desde 24/07/2026.

**Bug corrigido nesta rodada:** `dim_orgao.sk_orgao = md5(codigo, ano)` muda a cada ano pro mesmo órgão físico. O painel agora agrupa por `codigo_orgao` (estável), não por `sk_orgao` — sem isso, o histórico plurianual de cada órgão ficava invisível pro modelo (ver nota detalhada na célula de topo do notebook).

### 2.1 Extração — série de ordens bancárias por órgão/trimestre + vigência de contratos

In [ ]:
pagamentos = pf.extract_pagamento_series()
contratos = pf.extract_contratos_vigencia()
print(f"Linhas de pagamento (OB): {len(pagamentos)} · órgãos distintos: {pagamentos['codigo_orgao'].nunique()}")
print(f"Contratos com vigência: {len(contratos)}")
pagamentos.head()


### 2.2 Painel trimestral (órgão × trimestre)

`build_quarterly_panel` agrega o valor por trimestre, calcula `valor_contratado_ativo` (soma de contratos vigentes naquele trimestre, contando contratos assinados em QUALQUER ano — correção do bug), a flag de ano eleitoral e os lags (`lag_1_trimestre`, `lag_4_trimestres`, agora capazes de atravessar virada de ano). O alvo (`target_proximo_trimestre`) é o valor do **próximo** trimestre do mesmo órgão.

In [ ]:
panel = pf.build_quarterly_panel(pagamentos, contratos)
print(f"Linhas do painel: {len(panel)}")
print(f"lag_4_trimestres com valor real (não-fallback): {panel['lag_4_trimestres'].notna().sum()} / {len(panel)}"
      f" ({panel['lag_4_trimestres'].notna().mean():.1%})")
print(f"valor_contratado_ativo == 0: {(panel['valor_contratado_ativo'] == 0).sum()} / {len(panel)}"
      f" ({(panel['valor_contratado_ativo'] == 0).mean():.1%})")
panel.sort_values(["codigo_orgao", "ano", "trimestre"]).head(10)


In [ ]:
print("Trimestres cobertos:")
print(panel[["ano", "trimestre"]].drop_duplicates().sort_values(["ano", "trimestre"]))


### 2.3 Matriz de features

In [ ]:
X_pf = pf.build_feature_matrix(panel)
y_pf = panel.loc[X_pf.index, "target_proximo_trimestre"]
print(f"Shape da matriz de features: {X_pf.shape}")
print(f"Linhas com alvo conhecido (treináveis): {y_pf.notna().sum()}")
X_pf.describe().T


### 2.4 Avaliação — holdout temporal

O último trimestre com alvo conhecido vira teste; o resto treina — evita vazamento (nunca treina com dado "do futuro" em relação ao holdout). Métricas: MAE da mediana (p50) e cobertura do intervalo [p10, p90], que deveria rondar 80% se os quantis estiverem bem calibrados.

In [ ]:
metrics = pf.evaluate(panel, X_pf)
metrics


### 2.5 Modelo final — treinado com todo o histórico conhecido

In [ ]:
train_idx_pf = X_pf.index[y_pf.notna()]
models_pf = pf.train_models(X_pf.loc[train_idx_pf], y_pf.loc[train_idx_pf])
preds_treino = pf.predict_quantiles(models_pf, X_pf.loc[train_idx_pf])
print("Amplitude média do intervalo (p90 - p10):", (preds_treino["p90"] - preds_treino["p10"]).mean())


### 2.6 Previsão do próximo trimestre por órgão

Aplica o modelo às linhas cujo alvo ainda não existe (o trimestre mais recente de cada órgão) — a previsão real pedida pelo enunciado.

In [ ]:
resultado_pf = pf.forecast_next_quarter(panel, X_pf, models_pf)
print(f"Órgãos com previsão: {len(resultado_pf)}")
resultado_pf.sort_values("valor_previsto_p50", ascending=False).head(20)


### 2.7 Histórico vs. previsão — exemplo para um órgão

In [ ]:
NOME_DO_ORGAO = resultado_pf["nome_orgao"].iloc[0] if not resultado_pf.empty else None

if NOME_DO_ORGAO:
    import matplotlib.pyplot as plt

    hist_orgao = panel[panel["nome_orgao"] == NOME_DO_ORGAO].sort_values(["ano", "trimestre"])
    prev_orgao = resultado_pf[resultado_pf["nome_orgao"] == NOME_DO_ORGAO].iloc[0]
    prev_ds = pd.Timestamp(f"{int(prev_orgao['ano_previsto'])}-{(int(prev_orgao['trimestre_previsto']) - 1) * 3 + 1:02d}-01")
    hist_ds = pd.to_datetime(
        hist_orgao["ano"].astype(str) + "-" + ((hist_orgao["trimestre"] - 1) * 3 + 1).astype(str) + "-01"
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(hist_ds, hist_orgao["valor_trimestre"], marker="o", label="histórico (valor pago)")
    ax.errorbar(
        [prev_ds], [prev_orgao["valor_previsto_p50"]],
        yerr=[[prev_orgao["valor_previsto_p50"] - prev_orgao["valor_previsto_p10"]],
              [prev_orgao["valor_previsto_p90"] - prev_orgao["valor_previsto_p50"]]],
        color="red", capsize=5, marker="s", label="previsão (mediana + intervalo p10-p90)",
    )
    ax.set_title(f"Pagamentos por trimestre — {NOME_DO_ORGAO}")
    ax.set_ylabel("valor (R$)")
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum órgão teve histórico suficiente para gerar previsão.")


### 2.8 Persistência do modelo

Salva os 3 regressores (p10/p50/p90) em `models/artifacts/` — reproduzível a partir deste notebook ou de `python -m models.payment_forecast`.

In [ ]:
pf.save_model(models_pf, list(X_pf.columns))
print(f"Modelo salvo em: {pf.ARTIFACT_PATH}")


### Observações para o relatório final — Modelo 2

- [ ] **Órgãos com previsão gerada** — rode a célula 2.6 e preencha com o total desta execução.
- [x] **Bug do `sk_orgao` versionado por ano — corrigido (25/07/2026).** Achado comparando com o script independente da Fernanda (que usa Prophet e tinha corrigido o mesmo problema por outro caminho). Impacto medido em produção antes da correção: `lag_4_trimestres` **0 de 14.920 linhas** com valor real (sempre caía no fallback pra `lag_1_trimestre` — a feature de sazonalidade ano-a-ano nunca funcionou); `valor_contratado_ativo` **zerado em 89,5% das linhas** (contratos plurianuais só contavam como ativos no ano em que foram assinados). Corrigido agrupando por `codigo_orgao` em toda a pipeline — ver célula 2.2 acima pros números pós-correção.
- [x] **Fonte de "ordens bancárias" confirmada** — `iceberg.gold.fato_ordem_bancaria` (não é mais proxy via `fato_empenho` ou `fato_contrato.valor_pago`).
- [ ] Confirmar se o intervalo de confiança (célula 2.5) está numa amplitude razoável, ou se indica poucos dados / alta volatilidade real do órgão.

## 3. Componente de IA Generativa — Relatório Narrativo

`models/narrative_report.py` lê os dois resultados acima (`score_anomalia_contrato` + `previsao_pagamento_orgao`, já gravados na Gold) via Trino, monta um prompt só com os números já calculados (o LLM não recebe dado bruto nem infere valor novo) e usa a API OpenAI (`gpt-4o-mini` por padrão) para escrever um relatório em linguagem acessível para um gestor público sem formação técnica.

Requer `OPENAI_API_KEY` configurada no `.env`.

In [ ]:
top_anomalias = nr.extract_top_anomalias(top_n=10)
top_previsoes = nr.extract_top_previsoes(top_n=10)
print(f"Contratos atípicos considerados: {len(top_anomalias)} | Previsões consideradas: {len(top_previsoes)}")


In [ ]:
relatorio = nr.generate_narrative(top_anomalias, top_previsoes)
print(relatorio)


### Observações para o relatório final — IA Generativa

- [ ] Ler o relatório gerado na célula acima — o texto evita jargão técnico ("score", "quantil") e trata atipicidade como sinal para revisão humana, não acusação?
- [ ] Revisar tom/curadoria do que entra no relatório com o Benjamim (dono da tarefa 25) — adiantada pelo Jaime, ainda sem revisão do dono original.

## 4. Resumo para o relatório final

- **Modelo 1** — Isolation Forest sobre `fato_contrato` + `dim_credor`/`dim_modalidade` (Gold) e `tipo_objeto`/vigência (Silver). Score contínuo + `flag_anomalia` binária. Persistido em `iceberg.gold.score_anomalia_contrato`.
- **Modelo 2** — XGBoost por quantil sobre `fato_ordem_bancaria` (pagamento real, não proxy). Previsão + intervalo de confiança por órgão. Persistido em `iceberg.gold.previsao_pagamento_orgao`.
- **IA Generativa** — LLM (API OpenAI) lê os dois resultados acima e escreve um relatório narrativo em `iceberg.gold.relatorio_narrativo` + arquivo Markdown.
- **2 bugs reais corrigidos nesta rodada** (25/07/2026), achados comparando com o script independente da Fernanda:
  1. `dim_credor.historico_infringement` 100% `False` por dedup errado (`scd_credor.sql`).
  2. `sk_orgao` versionado por ano quebrando o histórico plurianual do Modelo 2 (`payment_forecast.py`).
- **Limitações conhecidas:**
  - Modelo 1: sem rótulo ground-truth — avaliação é qualitativa (distribuição + checagem de sanidade), recomendado validar com analistas de controle (tarefa #35).
  - Modelo 2: exige `lag_1_trimestre` (pelo menos 1 trimestre de histórico anterior) para entrar no treino/previsão — órgãos muito novos ficam de fora até acumular histórico.
  - IA Generativa: qualidade do texto depende do modelo de LLM configurado (`OPENAI_MODEL`, `gpt-4o-mini` por padrão) — ainda sem revisão humana formal do tom/curadoria.
- Todos os três rodam automaticamente em produção via DAG `dags/dag_ml_inference.py` (`score_anomalias`, `prever_pagamentos`, `gerar_relatorio_narrativo`) — este notebook é só para treino/avaliação exploratórios, não escreve na Gold por si (a não ser que as células de `save_model` acima sejam rodadas, que só afetam o `.joblib` local, não a Gold).